In [78]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [79]:
import os
os.chdir('/content/drive/MyDrive/Colab Notebooks/특화E205_AI_TEST')
!pwd
!ls

/content/drive/MyDrive/Colab Notebooks/특화E205_AI_TEST
config.py   mlruns    __pycache__	      runs	      train.py
dataset.py  model.py  requirements_colab.txt  server.py       uploads
index.html  MPIIGaze  requirements.txt	      TEST_RESULT.md


In [80]:
# requirements.txt에서 pywin32 줄 제거 후 설치
!grep -v "pywin32" requirements.txt > requirements_colab.txt
!pip install -r requirements_colab.txt

  Using cached protobuf-6.33.5-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
Using cached protobuf-6.33.5-cp39-abi3-manylinux2014_x86_64.whl (323 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.8
    Uninstalling protobuf-4.25.8:
      Successfully uninstalled protobuf-4.25.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.13 requires protobuf<5,>=4.25.3, but you have protobuf 6.33.5 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.

In [82]:
import torch
print(torch.cuda.is_available())   # True 이어야 함
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [83]:
%cd '/content/drive/MyDrive/Colab Notebooks/특화E205_AI_TEST'
!pwd
!ls

/content/drive/MyDrive/Colab Notebooks/특화E205_AI_TEST
/content/drive/MyDrive/Colab Notebooks/특화E205_AI_TEST
config.py   mlruns    __pycache__	      runs	      train.py
dataset.py  model.py  requirements_colab.txt  server.py       uploads
index.html  MPIIGaze  requirements.txt	      TEST_RESULT.md


In [84]:
!python train.py

  L2CS-Net 학습 시작
  디바이스       : cuda
  배치 크기      : 128
  학습률         : 0.0005
  에폭 수        : 30
  Bin 수         : 90
  손실 가중치    : CLS=1.0, REG=1.0

[1/5] 데이터 로딩 중...
[데이터 로드 완료] 총 480개 샘플
  이미지 shape: (480, 36, 60)
  Yaw 범위:  [-16.4°, 15.5°]
  Pitch 범위: [-15.9°, 0.2°]
[DataLoader 생성 완료] 학습: 384개 / 검증: 96개

[2/5] L2CS-Net 모델 생성 중...
  학습 가능 파라미터: 23,876,852개

[3/5] 손실함수 & 옵티마이저 설정...

[4/5] 체크포인트 저장 경로: /content/drive/MyDrive/Colab Notebooks/특화E205_AI_TEST/runs/exp15

[5/5] MLflow 실험 설정: 'L2CS-Net_MPIIGaze_p00'
/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)

  학습 루프 시작

[Epoch 01/30] T

In [85]:
import mlflow

mlflow_uri = "file:///content/drive/MyDrive/Colab Notebooks/특화E205_AI_TEST/mlruns"
mlflow.set_tracking_uri(mlflow_uri)

runs = mlflow.search_runs()
print(runs.columns.tolist())  # 실제 컬럼명 확인

['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time', 'end_time']


In [86]:
# 컬럼명 확인 후 존재하는 것만 선택
display(runs)  # 전체 DataFrame 표시

,run_id,experiment_id,status,artifact_uri,start_time,end_time


In [87]:
import mlflow

mlflow_uri = "file:///content/drive/MyDrive/Colab Notebooks/특화E205_AI_TEST/mlruns"
mlflow.set_tracking_uri(mlflow_uri)

# 실험 목록 확인
client = mlflow.MlflowClient()
experiments = client.search_experiments()
for exp in experiments:
    print(f"ID: {exp.experiment_id}, Name: {exp.name}")

ID: 977907538411323958, Name: L2CS-Net_MPIIGaze_p00
ID: 0, Name: Default


In [88]:
# 위에서 나온 experiment_id로 교체 (예: "1" 또는 "977907538411323958")
runs = mlflow.search_runs(experiment_ids=["977907538411323958"])
print(runs.columns.tolist())
display(runs)

['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time', 'end_time', 'metrics.val_angular_error', 'metrics.train_yaw_acc', 'metrics.train_cls_loss', 'metrics.train_angular_error', 'metrics.train_loss', 'metrics.val_loss', 'metrics.train_pitch_acc', 'metrics.val_pitch_acc', 'metrics.val_cls_loss', 'metrics.train_reg_loss', 'metrics.val_yaw_acc', 'metrics.val_reg_loss', 'params.alpha_cls', 'params.batch_size', 'params.num_epochs', 'params.alpha_reg', 'params.learning_rate', 'params.num_bins', 'params.image_size', 'params.weight_decay', 'params.bin_width', 'params.optimizer', 'params.backbone', 'params.use_both_eyes', 'tags.mlflow.user', 'tags.mlflow.runName', 'tags.mlflow.source.name', 'tags.mlflow.source.type']


,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.val_angular_error,metrics.train_yaw_acc,metrics.train_cls_loss,metrics.train_angular_error,...,params.image_size,params.weight_decay,params.bin_width,params.optimizer,params.backbone,params.use_both_eyes,tags.mlflow.user,tags.mlflow.runName,tags.mlflow.source.name,tags.mlflow.source.type
0,9e2db00c5ba14d92be7e6b1f87d2a12c,977907538411323958,FINISHED,file:///C:/Users/SSAFY/Desktop/L2CS-NET_test_l...,2026-03-04 12:56:40.662000+00:00,2026-03-04 13:01:20.618000+00:00,1.512218,0.500000,2.815465,0.635164,...,224,0.0001,4.0,AdamW,ResNet50,True,root,exp15,train.py,LOCAL
1,5c912fe99a294dc3bcd71bc8e40ba506,977907538411323958,FINISHED,file:///C:/Users/SSAFY/Desktop/L2CS-NET_test_l...,2026-03-04 11:39:32.311000+00:00,2026-03-04 11:44:07.732000+00:00,2.009232,0.518229,3.295790,0.606189,...,224,0.0001,4.0,AdamW,ResNet50,True,root,exp14,train.py,LOCAL
2,80f0b038c0124f7e9a6334625b26cc9b,977907538411323958,FINISHED,file:///C:/Users/SSAFY/Desktop/L2CS-NET_test_l...,2026-03-04 10:50:42.890000+00:00,2026-03-04 10:56:05.989000+00:00,1.653351,0.447917,2.992618,0.512893,...,224,0.0001,4.0,AdamW,ResNet50,True,root,exp13,train.py,LOCAL
3,8e8e0820795848acb81a347e20ec1ae5,977907538411323958,FINISHED,file:///C:/Users/SSAFY/Desktop/L2CS-NET_test_l...,2026-03-04 10:45:04.945000+00:00,2026-03-04 10:46:50.492000+00:00,7.375922,0.208333,8.559604,0.826487,...,224,0.0001,4.0,AdamW,ResNet50,False,root,exp12,train.py,LOCAL
4,867fec71b69c4760bae0e729ff5c3b84,977907538411323958,FINISHED,file:///C:/Users/SSAFY/Desktop/L2CS-NET_test_l...,2026-03-04 10:30:49.912000+00:00,2026-03-04 10:31:06.548000+00:00,8.783569,0.000000,9.042426,7.371182,...,224,0.0001,4.0,AdamW,ResNet50,False,root,exp11,train.py,LOCAL
5,95a72247c37e4370a8141ff47ef6c8fd,977907538411323958,RUNNING,file:///C:/Users/SSAFY/Desktop/L2CS-NET_test_l...,2026-03-04 08:19:42.577000+00:00,NaT,NaN,NaN,NaN,NaN,...,224,0.0001,4.0,AdamW,ResNet50,False,SSAFY,exp10,train.py,LOCAL
6,ba956c2a700a4aefb98605597663f89e,977907538411323958,FAILED,file:///C:/Users/SSAFY/Desktop/L2CS-NET_test_l...,2026-03-04 07:51:22.298000+00:00,2026-03-04 08:19:30.864000+00:00,NaN,NaN,NaN,NaN,...,224,0.0001,4.0,AdamW,ResNet50,False,SSAFY,exp9,train.py,LOCAL
7,f120992b9a0c4f9298a97f8f091e0d22,977907538411323958,FAILED,file:///C:/Users/SSAFY/Desktop/L2CS-NET_test_l...,2026-03-04 07:44:38.310000+00:00,2026-03-04 07:51:12.307000+00:00,NaN,NaN,NaN,NaN,...,224,0.0001,4.0,AdamW,ResNet50,False,SSAFY,exp8,train.py,LOCAL
8,79bec31dcb9b4340880dc390df96b67e,977907538411323958,FAILED,file:///C:/Users/SSAFY/Desktop/L2CS-NET_test_l...,2026-03-04 07:40:55.401000+00:00,2026-03-04 07:44:24.947000+00:00,NaN,NaN,NaN,NaN,...,224,0.0001,4.0,AdamW,ResNet50,False,SSAFY,exp7,train.py,LOCAL
9,b5ee7e9797b44526980e5cc078417913,977907538411323958,FAILED,file:///C:/Users/SSAFY/Desktop/L2CS-NET_test_l...,2026-03-04 07:28:08.819000+00:00,2026-03-04 07:37:50.463000+00:00,NaN,NaN,NaN,NaN,...,224,0.0001,4.0,AdamW,ResNet50,False,SSAFY,exp6,train.py,LOCAL


In [58]:
!pip install mediapipe -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 15.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatible.


In [89]:
!pip install mediapipe -q

import subprocess, time

# stderr를 파일로 리다이렉트해서 오류 확인
proc = subprocess.Popen(
    ["python", "server.py"],
    cwd='/content/drive/MyDrive/Colab Notebooks/특화E205_AI_TEST', # 수정된 부분
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(5)

# 서버가 살아있는지 확인
if proc.poll() is not None:
    # 이미 죽어있음 → 오류 출력
    out, err = proc.communicate()
    print("STDOUT:", out.decode())
    print("STDERR:", err.decode())
else:
    print("서버 정상 실행 중")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.39.1 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-common 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.39.1 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.
google-adk 1.26.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you hav

In [90]:
!pip install mediapipe==0.10.13 -q

In [76]:
import subprocess, time

proc = subprocess.Popen(
    ["python", "server.py"],
    cwd='/content/drive/MyDrive/Colab Notebooks/특화E205_AI_TEST',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT  # stdout+stderr 합치기
)

# 15초간 출력 실시간 확인
for _ in range(30):
    line = proc.stdout.readline().decode(errors='replace')
    if line:
        print(line, end='')
    if proc.poll() is not None:
        # 프로세스 종료됨 → 남은 출력 전부 출력
        remaining = proc.stdout.read().decode(errors='replace')
        print(remaining)
        print("=== 서버 종료됨 ===")
        break
    time.sleep(0.5)

2026-03-04 12:15:04.569404: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772626504.590980   30555 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772626504.598054   30555 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772626504.616319   30555 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772626504.616339   30555 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772626504.616341   30555 computation_placer.cc:177] computation placer alr

In [91]:
import subprocess, time, re

# 서버 시작
server_proc = subprocess.Popen(
    ["python", "server.py"],
    cwd='/content/drive/MyDrive/Colab Notebooks/특화E205_AI_TEST',
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(8)

if server_proc.poll() is not None:
    _, err = server_proc.communicate()
    print("서버 크래시:", err.decode())
else:
    print("서버 정상 실행 중")

    # cloudflared 설치 및 터널 실행
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
        shell=True
    )

    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )

    # URL 추출 (최대 15초 대기)
    for _ in range(30):
        line = tunnel_proc.stdout.readline().decode()
        match = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
        if match:
            print(f"접속 URL: {match.group()}")
            break
        time.sleep(0.5)

서버 정상 실행 중
접속 URL: https://richard-images-adaptation-overnight.trycloudflare.com


In [68]:
from pyngrok import ngrok

# 기존 터널 전부 종료
ngrok.kill()

import time
time.sleep(2)

# 다시 연결
public_url = ngrok.connect("127.0.0.1:8000")
print(f"접속 URL: {public_url}")

접속 URL: NgrokTunnel: "https://skye-noncarbonated-nonvariably.ngrok-free.dev" -> "http://127.0.0.1:8000"
